In [1]:
from google.colab import files
uploaded = files.upload()

Saving 2026-02-17_exp.csv to 2026-02-17_exp.csv
Saving 2026-02-24_exp.csv to 2026-02-24_exp.csv
Saving 2026-03-02_exp.csv to 2026-03-02_exp.csv


In [2]:
!pip install pandas numpy scikit-learn plotly

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import IsolationForest

In [4]:
df1 = pd.read_csv("/2026-02-17_exp.csv")
df2 = pd.read_csv("/2026-02-24_exp.csv")
df3 = pd.read_csv("/2026-03-02_exp.csv")

df = pd.concat([df1, df2, df3], ignore_index=True)

df.head()

,symbol,datetime,expiry,CE,PE,spot_close,ATM,strike,oi_CE,oi_PE,volume_CE,volume_PE
0,NIFTY,2026-02-10 15:13:00,2026-02-17,2855.55,2.25,25933.30,25950.0,23100,65,2496390,130,67600
1,NIFTY,2026-02-10 15:12:00,2026-02-17,2811.30,2.30,25937.85,25950.0,23150,65,178555,65,1430
2,NIFTY,2026-02-10 15:13:00,2026-02-17,2756.70,2.45,25933.30,25950.0,23200,325,496015,130,25350
3,NIFTY,2026-02-10 15:26:00,2026-02-17,2730.25,2.50,25915.15,25900.0,23200,325,749515,130,45955
4,NIFTY,2026-02-10 15:10:00,2026-02-17,2665.70,2.45,25941.20,25950.0,23300,650,233025,130,8905


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
df['datetime'] = pd.to_datetime(df['datetime'])

In [9]:
df = df.sort_values("datetime")

In [11]:
df["put_call_ratio"] = df["volume_PE"] / df["volume_CE"]

In [13]:
df["total_oi"] = df["oi_CE"] + df["oi_PE"]

In [15]:
volume_ts = df.groupby("datetime")[["volume_CE","volume_PE"]].sum().reset_index()

fig = px.line(volume_ts, x="datetime", y=["volume_CE","volume_PE"],
              title="Options Volume Over Time")

fig.show()

In [17]:
strike_data = df.groupby("strike")[["oi_CE","oi_PE"]].sum().reset_index()

fig = px.bar(strike_data, x="strike", y=["oi_CE","oi_PE"],
             title="Open Interest by Strike")

fig.show()

In [19]:
vol = df.groupby("strike")["CE"].mean().reset_index()

fig = px.line(vol, x="strike", y="CE",
              title="Call Price by Strike (Implied Volatility Missing)")

fig.show()

In [21]:
features = df[["volume_CE","volume_PE","oi_CE","oi_PE"]]

model = IsolationForest(contamination=0.01)

df["anomaly"] = model.fit_predict(features)

In [22]:
anomalies = df[df["anomaly"] == -1]

anomalies.head()

,symbol,datetime,expiry,CE,PE,spot_close,ATM,strike,oi_CE,oi_PE,volume_CE,volume_PE,put_call_ratio,total_oi,anomaly
44804,NIFTY,2026-02-16 09:31:00,2026-02-17,86.40,85.20,25500.00,25500.0,25500,17445090,15282735,7081880,3691155,0.521211,32727825,-1
44824,NIFTY,2026-02-16 09:51:00,2026-02-17,62.95,109.80,25440.30,25450.0,25500,20783685,17462055,3602105,4824560,1.339372,38245740,-1
44833,NIFTY,2026-02-16 10:00:00,2026-02-17,82.30,81.15,25499.35,25500.0,25500,22271340,15832505,7099430,4133155,0.582181,38103845,-1
44834,NIFTY,2026-02-16 10:01:00,2026-02-17,91.80,74.65,25515.85,25500.0,25500,21301670,17429945,5893290,3596385,0.610251,38731615,-1
44835,NIFTY,2026-02-16 10:02:00,2026-02-17,95.45,72.10,25522.35,25500.0,25500,21301670,17429945,6383130,4216160,0.660516,38731615,-1


In [24]:
fig = px.scatter(df,
                 x="datetime",
                 y="volume_CE",
                 color="anomaly",
                 title="Anomaly Detection in Options Volume")

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [25]:
df["market_sentiment"] = np.where(df["put_call_ratio"] > 1,
                                  "Bearish",
                                  "Bullish")

In [26]:
df.to_csv("processed_options_data.csv", index=False)

In [27]:
files.download("processed_options_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>